# Wind Power Forecasting with LLMs - Results Replication

This notebook replicates the results from the Case Study Report:
- **Table 2**: Performance across horizons (3h, 6h, 48h) for different tokenization methods
- **Table 3**: Performance across different LLM models and prompting strategies

**Authors**: Abdulwahab Albassam, Aidan Leung, Jett Ngo

**Dataset**: SDWPF (Spatial-Dynamic Wind Power Forecasting) from Baidu KDD Cup 2022

## Setup and Configuration

In [ ]:
import os
import json
import time
import re
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Weather API imports
import requests
try:
    import openmeteo_requests
    import requests_cache
    from retry_requests import retry
    WEATHER_API_AVAILABLE = True
except ImportError:
    print("⚠️ Weather API libraries not installed. Install with: pip install openmeteo-requests requests-cache retry-requests")
    WEATHER_API_AVAILABLE = False

### API Configuration

Choose your LLM provider and set API keys:

In [ ]:
# ========================================
# CONFIGURATION: Choose your LLM provider
# ========================================

PROVIDER = "gemini"  # Options: "claude" or "gemini"
MODEL_NAME = "gemini-3-flash"  # Will be updated based on provider

# API Keys - Set these as environment variables or paste directly (not recommended for production)
CLAUDE_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "YOUR_CLAUDE_KEY_HERE")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_KEY_HERE")

# Setup based on provider
if PROVIDER == "claude":
    import anthropic
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    MODEL_NAME = "claude-3-haiku-20240307"
    print(f"✅ Using Claude: {MODEL_NAME}")
    
elif PROVIDER == "gemini":
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    client = genai.GenerativeModel('models/gemini-3-flash-preview')
    MODEL_NAME = "gemini-3-flash"
    print(f"✅ Using Gemini: {MODEL_NAME}")
    print("   ⚠️  Note: Free tier has rate limits (20 requests/day)")
else:
    raise ValueError(f"Unknown provider: {PROVIDER}")

## Helper Functions

### Evaluation Metrics

In [ ]:
def evaluate_forecast(df, Turb_ID, base_day, output, horizon_hours=48):
    """
    Evaluates LLM forecast against ground truth.
    
    Args:
        df: DataFrame with SCADA data
        Turb_ID: Turbine ID to evaluate
        base_day: Starting day of historical window
        output: LLM response (JSON string or raw numbers)
        horizon_hours: Forecast horizon in hours (3, 6, or 48)
    
    Returns:
        Dictionary with MAE, RMSE, Score, and metadata
    """
    # Calculate expected number of points (6 per hour for 10-min intervals)
    expected_points = horizon_hours * 6
    
    # Define target window (starts after 14-day history)
    target_start = base_day + 14
    target_days = int(np.ceil(horizon_hours / 24))
    target_end = target_start + target_days - 1
    
    # Extract ground truth
    actual_data = df[
        (df['TurbID'] == Turb_ID) &
        (df['Day'] >= target_start) &
        (df['Day'] <= target_end)
    ]
    
    actual_values = (
        actual_data
        .sort_values(['Day', 'Tmstamp'])
        ['Patv']
        .dropna()
        .tolist()
    )[:expected_points]
    
    # Parse LLM output
    try:
        # Try JSON parsing first
        clean_json = output.replace('```json', '').replace('```', '').strip()
        data = json.loads(clean_json)
        predicted_values = data.get('forecast', [])
    except:
        # Fallback: extract numbers using regex
        predicted_values = [float(x) for x in re.findall(r"[-+]?\d*\.\d+|\d+", output)]
    
    # Clip to expected length
    predicted_values = predicted_values[:expected_points]
    
    # Validation
    if len(predicted_values) != expected_points:
        print(f"⚠️ Warning: Expected {expected_points} predictions, got {len(predicted_values)}")
    
    if len(actual_values) != expected_points:
        print(f"⚠️ Warning: Expected {expected_points} actual points, got {len(actual_values)}")
    
    # Align lengths
    min_len = min(len(actual_values), len(predicted_values))
    
    if min_len == 0:
        print("❌ Error: No overlapping forecast points")
        return None
    
    y_true = actual_values[:min_len]
    y_pred = predicted_values[:min_len]
    
    # Calculate metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    score = (mae + rmse) / 2
    
    print(f"\n--- {horizon_hours}h Evaluation (Days {target_start}-{target_end}) ---")
    print(f"Points used: {min_len}/{expected_points}")
    print(f"MAE:  {mae:.2f} kW")
    print(f"RMSE: {rmse:.2f} kW")
    print(f"Overall Score: {score:.2f} kW")
    
    return {
        "MAE": mae,
        "RMSE": rmse,
        "Score": score,
        "points_used": min_len,
        "horizon_hours": horizon_hours
    }

### Data Binning (for APBF strategy)

In [ ]:
def bin_turbine_data(df, n_bins=16, bin_cols=None):
    """
    Bins numeric columns into equal-width integer IDs [0, n_bins-1].
    This reduces token count for LLM input.
    """
    hist = df.copy()
    if bin_cols is None:
        excluded = {"TurbID", "Day", "Tmstamp"}
        bin_cols = [
            c for c in hist.columns 
            if (c not in excluded) and pd.api.types.is_numeric_dtype(hist[c])
        ]
    
    for c in bin_cols:
        x = pd.to_numeric(hist[c], errors="coerce")
        valid = x.dropna()
        if valid.empty:
            continue
        
        lo, hi = float(valid.min()), float(valid.max())
        
        if np.isclose(lo, hi):
            hist[c] = 0
            continue
        
        # Create equal-width bins
        edges = np.linspace(lo, hi, n_bins + 1)
        binned = pd.cut(
            x,
            bins=edges,
            labels=False,
            include_lowest=True,
            duplicates="drop",
        )
        hist[c] = binned.astype("Int64")
    
    return hist, bin_cols

### Weather Forecast Tool (Open-Meteo API)

In [ ]:
def get_openmeteo_wind_data(lat, lon, start_date, end_date, max_retries=3):
    """
    Fetches 100m wind speed data from Open-Meteo Archive API (ERA5 reanalysis).
    """
    if not WEATHER_API_AVAILABLE:
        raise ImportError("Weather API libraries not installed")
    
    # Setup the API client with caching and retry logic
    cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
    retry_session = retry(cache_session, retries=max_retries, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "wind_speed_100m",
        "timezone": "auto",
        "wind_speed_unit": "ms",
    }
    
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    
    # Process hourly data
    hourly = response.Hourly()
    hourly_wind_speed_100m = hourly.Variables(0).ValuesAsNumpy()
    
    hourly_data = {
        "timestamp": pd.date_range(
            start=pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        ),
        "wind_speed_100m": hourly_wind_speed_100m
    }
    
    return pd.DataFrame(data=hourly_data)

## Prompting Strategies

Implementing the three main strategies from the report:
1. **Naive Baseline**: Simple prompting with raw SCADA data
2. **Advanced Prompt**: Physics-informed prompting
3. **APBF (Advanced Prompt + Binning + Forecast)**: Full pipeline with ERA5 data

### Strategy 1: Naive Baseline

In [ ]:
def call_llm_naive(df, Turb_ID, base_day, horizon_hours=48):
    """
    Naive baseline: Simple prompt with raw SCADA data.
    No preprocessing, no weather forecasts, minimal instructions.
    """
    turbine_data = df[df['TurbID'] == Turb_ID]
    turbine_data = turbine_data.dropna(subset=['Wspd'])
    
    end_day = base_day + 13
    window_data = turbine_data[
        (turbine_data['Day'] >= base_day) & 
        (turbine_data['Day'] <= end_day)
    ].drop(columns=['TurbID'])
    
    data_str = window_data.to_csv(index=False, sep=',')
    expected_points = horizon_hours * 6
    
    prompt = f"""You are given the past 14 days of 10-minute SCADA data.
Columns: Day, Tmstamp, Wspd, Etmp, Itmp, Patv

Forecast Patv (kW) for Day {end_day + 1} to Day {end_day + int(np.ceil(horizon_hours/24))} at 10-minute resolution.
Return a csv file with one column only.

Data:
{data_str}
"""
    
    # Call LLM
    if PROVIDER == "claude":
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=2048,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    else:  # gemini
        response = client.generate_content(prompt)
        return response.text

### Strategy 2: Advanced Prompt (Physics-Informed)

In [ ]:
def call_llm_advanced(df, Turb_ID, base_day, horizon_hours=48):
    """
    Advanced prompting strategy with:
    - Physics-informed constraints (P ∝ W³)
    - Explicit output format requirements
    - Domain knowledge injection
    """
    turbine_data = df[df['TurbID'] == Turb_ID]
    turbine_data = turbine_data.dropna(subset=['Wspd'])
    
    end_day = base_day + 13
    window_data = turbine_data[
        (turbine_data['Day'] >= base_day) & 
        (turbine_data['Day'] <= end_day)
    ].copy()
    
    data_str = window_data.to_csv(index=False, sep=',')
    expected_points = horizon_hours * 6
    
    prompt = f"""Context: You are an expert wind turbine power forecasting model.

You are given 14 days of historical SCADA data for ONE turbine.
The data is sampled every 10 minutes (144 rows per day).
Columns: {', '.join(window_data.columns.tolist())}

Input Data:
{data_str}

Your task:
Predict the Active Power (Patv, in kW) for the NEXT {horizon_hours} HOURS
({expected_points} time steps at 10-minute resolution).

Instructions:

Learn the wind-speed to power relationship from the historical data.
Power is roughly proportional to wind speed cubed at low speeds.
Power saturates near rated power (~1500 kW).
Power is near zero at very low wind speeds.

Capture daily cyclic patterns.
Do NOT copy the last day.
Do NOT output negative values.
Clip values to the realistic range [0, 1500].

OUTPUT FORMAT REQUIREMENTS (VERY IMPORTANT):

Return valid JSON in the following format:

{{
  "forecast": [f1, f2, f3, ..., f{expected_points}]
}}

The list MUST contain exactly {expected_points} numbers.
If the list contains more or fewer values, the answer is invalid.

No text outside the JSON.
No explanations.
No markdown formatting.
Only raw JSON.
"""
    
    # Call LLM
    if PROVIDER == "claude":
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=4096,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text
    else:  # gemini
        response = client.generate_content(prompt)
        return response.text

### Strategy 3: APBF (Advanced + Binning + Forecast)

In [ ]:
def call_llm_apbf(df, Turb_ID, base_day, horizon_hours=48, use_weather=True, max_retries=3):
    """
    Full APBF pipeline:
    - Binning for token reduction
    - ERA5 weather forecasts
    - Advanced physics-informed prompting
    - Retry logic for robustness
    """
    turbine_data = df[df['TurbID'] == Turb_ID]
    turbine_data = turbine_data.dropna(subset=['Wspd'])
    
    end_day = base_day + 13
    window_data = turbine_data[
        (turbine_data['Day'] >= base_day) & 
        (turbine_data['Day'] <= end_day)
    ].copy()
    
    # Apply binning
    print(f"📊 Binning data into 16 levels...")
    window_data, binned_list = bin_turbine_data(window_data, n_bins=16, bin_cols=["Wspd", "Patv"])
    bin_msg = f"Note: Numeric columns ({', '.join(binned_list)}) have been binned into 16 equal-width levels (0-15)."
    print(bin_msg)
    
    data_str = window_data.to_csv(index=False, sep=',')
    expected_points = horizon_hours * 6
    
    # Prepare weather forecast section
    forecast_section = "No external meteorological forecast provided."
    
    if use_weather and WEATHER_API_AVAILABLE:
        # Gansu Guazhou Qiaowan Number 3 North Wind Farm coordinates
        latitude = 40.6306
        longitude = 96.9498
        anchor_date = datetime(2020, 5, 1)  # Dataset start date
        forecast_start_date = anchor_date + timedelta(days=int(base_day + 13))
        forecast_end_date = forecast_start_date + timedelta(days=int(np.ceil(horizon_hours/24)))
        
        start_str = forecast_start_date.strftime('%Y-%m-%d')
        end_str = forecast_end_date.strftime('%Y-%m-%d')
        
        for attempt in range(1, max_retries + 1):
            try:
                print(f"🌐 Fetching weather forecast for {start_str} to {end_str} (attempt {attempt})...")
                weather_df = get_openmeteo_wind_data(latitude, longitude, start_str, end_str)
                
                # Interpolate hourly data to 10-minute intervals
                hourly_points = horizon_hours
                w_speeds = weather_df['wind_speed_100m'].values[:hourly_points]
                xp = np.arange(len(w_speeds))
                x_new = np.linspace(0, len(w_speeds)-1, expected_points)
                interp_w_speeds = np.interp(x_new, xp, w_speeds)
                
                forecast_str = ", ".join([f"{v:.2f}" for v in interp_w_speeds])
                forecast_section = f"""Input 2: {horizon_hours}-Hour Meteorological Forecast (100m)
Predicted wind speeds (m/s) at 10-minute intervals for the next {horizon_hours} hours:
[{forecast_str}]"""
                print(f"✅ Retrieved {len(interp_w_speeds)} forecast points")
                break
            except Exception as e:
                print(f"⚠️ Weather API failed (attempt {attempt}): {e}")
                if attempt < max_retries:
                    time.sleep(5)
                else:
                    print("❌ All weather API attempts failed. Proceeding with SCADA only.")
    
    # Construct prompt
    prompt = f"""Context: You are an expert wind turbine power forecasting model.

{bin_msg}

Input 1: You are given 14 days of historical SCADA data for ONE turbine.
The data is sampled every 10 minutes (144 rows per day).
Columns provided: {', '.join(window_data.columns.tolist())}
{data_str}

{forecast_section}

Your task:
Predict the Active Power (Patv, in kW) for the NEXT {horizon_hours} HOURS
({expected_points} time steps at 10-minute resolution).

Instructions:

Learn the wind-speed to power relationship from the historical data.
Power is roughly proportional to wind speed cubed at low speeds.
Power saturates near rated power (~1500 kW).
Power is near zero at very low wind speeds.

Capture daily cyclic patterns.
Do NOT copy the last day.
Do NOT output negative values.
Clip values to the realistic range [0, 1500].

OUTPUT FORMAT REQUIREMENTS (CRITICAL):

Return valid JSON in the following format:

{{
"forecast": [f1, f2, f3, ..., f{expected_points}]
}}

The list MUST contain exactly {expected_points} numbers.
If the list contains more or fewer values, the answer is invalid.

No text outside the JSON.
No explanations.
No markdown formatting.
Only raw JSON.
"""
    
    # LLM call with retry logic
    for llm_attempt in range(1, max_retries + 1):
        print(f"🤖 LLM inference attempt {llm_attempt}...")
        
        try:
            if PROVIDER == "claude":
                response = client.messages.create(
                    model=MODEL_NAME,
                    max_tokens=4096,
                    messages=[{"role": "user", "content": prompt}]
                )
                raw_text = response.content[0].text
            else:  # gemini
                response = client.generate_content(prompt)
                raw_text = response.text
            
            # Validate output
            clean_json = raw_text.replace('```json', '').replace('```', '').strip()
            data = json.loads(clean_json)
            
            if "forecast" in data and len(data["forecast"]) == expected_points:
                print(f"✅ Success: Received exactly {expected_points} values")
                return raw_text
            else:
                actual_len = len(data.get("forecast", []))
                print(f"⚠️ Validation failed: Expected {expected_points}, got {actual_len}")
        
        except Exception as e:
            print(f"⚠️ LLM call failed (attempt {llm_attempt}): {e}")
        
        if llm_attempt < max_retries:
            print("🔄 Retrying...")
            time.sleep(2)
    
    print("❌ All LLM attempts failed")
    return None

## Experimental Setup

Replicating the experimental configuration from Table 1 in the report.

In [ ]:
# Load dataset
print("Loading SDWPF dataset...")
df = pd.read_csv('wtbdata_245days.csv')
print(f"✅ Loaded {len(df)} rows, {df['TurbID'].nunique()} turbines, {df['Day'].nunique()} days")

# Experimental parameters (from report Table 1)
TURBINE_ID = 1  # Focus on Turbine 1 for consistency
BASE_DAY = 1    # Start from Day 1 (leaving room for 14-day history + 2-day forecast)
HORIZONS = [3, 6, 48]  # Hours (Ultra Short, Short, Long)
STRATEGIES = ['naive', 'advanced', 'apbf']  # Three prompting strategies

## Run Experiments

### Single Test Run (for debugging)

In [ ]:
# Test a single configuration to verify everything works
print("=" * 60)
print("Testing single configuration: 3h horizon, APBF strategy")
print("=" * 60)

test_output = call_llm_apbf(df, Turb_ID=TURBINE_ID, base_day=BASE_DAY, horizon_hours=3, use_weather=True)

if test_output:
    print("\n" + "=" * 60)
    print("LLM Response (first 200 chars):")
    print(test_output[:200])
    print("\n" + "=" * 60)
    
    test_results = evaluate_forecast(df, TURBINE_ID, BASE_DAY, test_output, horizon_hours=3)
    print("\n✅ Test completed successfully!")
else:
    print("\n❌ Test failed - check API keys and configuration")

### Full Experiment Suite (Table 2 Replication)

This will run all combinations to replicate Table 2 from the report:
- 3 horizons × 3 strategies = 9 experiments

⚠️ **Warning**: This can take 30-60 minutes depending on API rate limits!

In [ ]:
# Full experimental run
results = []

for strategy in STRATEGIES:
    for horizon in HORIZONS:
        print("\n" + "=" * 80)
        print(f"Running: {strategy.upper()} | {horizon}h horizon | Turbine {TURBINE_ID} | Day {BASE_DAY}")
        print("=" * 80)
        
        start_time = time.time()
        
        # Call appropriate strategy
        try:
            if strategy == 'naive':
                output = call_llm_naive(df, TURBINE_ID, BASE_DAY, horizon)
            elif strategy == 'advanced':
                output = call_llm_advanced(df, TURBINE_ID, BASE_DAY, horizon)
            elif strategy == 'apbf':
                output = call_llm_apbf(df, TURBINE_ID, BASE_DAY, horizon, use_weather=True)
            
            if output:
                eval_result = evaluate_forecast(df, TURBINE_ID, BASE_DAY, output, horizon)
                
                if eval_result:
                    results.append({
                        'Strategy': strategy,
                        'Horizon (h)': horizon,
                        'Model': MODEL_NAME,
                        'TurbID': TURBINE_ID,
                        'BaseDay': BASE_DAY,
                        'MAE': eval_result['MAE'],
                        'RMSE': eval_result['RMSE'],
                        'Overall': eval_result['Score'],
                        'Points': eval_result['points_used'],
                        'Runtime (s)': time.time() - start_time
                    })
                    print(f"✅ Completed in {time.time() - start_time:.1f}s")
            else:
                print("❌ LLM call failed")
                
        except Exception as e:
            print(f"❌ Error: {e}")
        
        # Rate limiting pause
        print("\n⏸️  Pausing 10s to respect API rate limits...")
        time.sleep(10)

# Save results
results_df = pd.DataFrame(results)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = f"replication_results_{timestamp}.csv"
results_df.to_csv(output_file, index=False)
print(f"\n💾 Results saved to: {output_file}")
print("\n" + "=" * 80)
print("FINAL RESULTS")
print("=" * 80)
print(results_df.to_string(index=False))

### Generate LaTeX Table (Table 2 format)

In [ ]:
# Pivot table to match report format
if len(results_df) > 0:
    pivot_table = results_df.pivot_table(
        index='Strategy',
        columns='Horizon (h)',
        values=['MAE', 'RMSE', 'Overall'],
        aggfunc='first'
    )
    
    print("\n" + "=" * 80)
    print("Table 2 Format: Performance Across Horizons")
    print("=" * 80)
    print(pivot_table.round(2))
    
    # LaTeX export
    latex_str = pivot_table.round(2).to_latex()
    print("\n" + "=" * 80)
    print("LaTeX Code:")
    print("=" * 80)
    print(latex_str)
else:
    print("No results to display")

## Visualization

In [ ]:
if len(results_df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, metric in enumerate(['MAE', 'RMSE', 'Overall']):
        ax = axes[idx]
        for strategy in STRATEGIES:
            subset = results_df[results_df['Strategy'] == strategy]
            if len(subset) > 0:
                ax.plot(subset['Horizon (h)'], subset[metric], marker='o', label=strategy.upper())
        
        ax.set_xlabel('Forecast Horizon (hours)')
        ax.set_ylabel(f'{metric} (kW)')
        ax.set_title(f'{metric} vs Horizon')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xscale('log')
    
    plt.tight_layout()
    plt.savefig(f'replication_results_{timestamp}.png', dpi=150)
    print(f"\n📊 Plot saved to: replication_results_{timestamp}.png")
    plt.show()
else:
    print("No results to plot")

## Compare with Original Results

Load and compare with the student's reported results from `wind_forecasting_results.csv`

In [ ]:
try:
    original_results = pd.read_csv('wind_forecasting_results.csv')
    print("Original Student Results:")
    print(original_results.head(10))
    
    # TODO: Add comparison logic here
    
except FileNotFoundError:
    print("⚠️ Original results file not found")

## Summary

This notebook replicates the key experiments from the Case Study:

**Key Findings from Original Report (Table 3):**
- Best model: **Gemini 3-flash (APBF)** with Overall Score: **241.07 kW**
- APBF significantly outperformed Naive baseline (~77% improvement)
- Binning + ERA5 forecasts + physics-informed prompting was crucial

**Reproduced Results:**
- (Results will appear above after running experiments)

**Notes:**
- Results may vary due to LLM non-determinism
- Weather data is from ERA5 reanalysis (historical forecasts not available for 2020)
- Free tier API limits may affect reproducibility